# 02 - Feature Engineering and Proxy Labels
## Surprise Medical Bill Risk Predictor - SDSU Capstone

This notebook:
1. Loads cleaned CMS data from Notebook 1
2. Engineers **40+ features** from raw financial and service data
3. Creates **proxy labels** for supervised learning
4. Outputs `data/model_features.parquet` for model training

### Feature Categories:
- **Financial ratios**: charge ratio, payment gap, blended metrics
- **Log transforms**: normalized financial magnitudes
- **Service flags**: ER, imaging, surgery, outpatient indicators
- **Provider aggregates**: provider-level risk patterns
- **State aggregates**: geographic risk patterns
- **Risk indices**: composite risk scores
- **Text features**: DRG description (for TF-IDF in model)

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

os.makedirs('data', exist_ok=True)
os.makedirs('artifacts', exist_ok=True)

# Load cleaned data
print('Loading cleaned data...')
df = pd.read_parquet('data/base_cleaned.parquet')
print(f'Loaded: {df.shape[0]:,} records, {df.shape[1]} columns')
print(df.head(3))

## 1. Financial Ratio Features

Charge ratios and payment gaps are the strongest predictors of surprise billing risk.

In [ ]:
eps = 1e-6

# --- INPATIENT FEATURES (from CMS IPPS data) ---
# charge_ratio: how much was billed vs. how much was paid (high ratio = more exposure)
df['charge_ratio_inpatient'] = df['average_covered_charges'] / (df['average_total_payments'] + eps)

# payment_gap: absolute dollar gap between what was billed vs paid
df['payment_gap_inpatient'] = df['average_covered_charges'] - df['average_total_payments']

# --- OUTPATIENT FEATURES (from CMS OPPS data) ---
df['charge_ratio_outpatient'] = df['Billed amount'] / (df['Medicare payment'] + eps)
df['payment_gap_outpatient'] = df['Billed amount'] - df['Medicare payment']

# --- BLENDED FEATURES (combined signal) ---
df['blended_charge_ratio'] = (
    0.6 * df['charge_ratio_inpatient'] +
    0.4 * df['charge_ratio_outpatient']
)
df['blended_payment_gap'] = (
    0.6 * df['payment_gap_inpatient'] +
    0.4 * df['payment_gap_outpatient']
)

print('Financial ratio features created:')
print(f'  charge_ratio_inpatient:  mean={df["charge_ratio_inpatient"].mean():.2f}, max={df["charge_ratio_inpatient"].max():.2f}')
print(f'  payment_gap_inpatient:   mean=${df["payment_gap_inpatient"].mean():,.0f}')
print(f'  charge_ratio_outpatient: mean={df["charge_ratio_outpatient"].mean():.2f}')
print(f'  blended_charge_ratio:    mean={df["blended_charge_ratio"].mean():.2f}')

## 2. Log-Transform Features

Log transforms reduce skewness in financial distributions and improve model stability.

In [ ]:
df['log_avg_covered']     = np.log1p(df['average_covered_charges'])
df['log_avg_payments']    = np.log1p(df['average_total_payments'])
df['log_billed_amount']   = np.log1p(df['Billed amount'])
df['log_medicare_payment']= np.log1p(df['Medicare payment'])
df['log_blended_gap']     = np.log1p(df['blended_payment_gap'].clip(lower=0))

print('Log-transform features created (5 features)')

## 3. Service Classification Flags

Service type flags encode clinical context — ER visits, imaging, and surgery are highest risk.

In [ ]:
# Emergency department visits (highest risk for out-of-network surprise bills)
df['is_er'] = df['service_type'].str.contains('emergency', case=False, na=False).astype(int)

# Imaging (radiology often billed separately by external groups)
df['is_imaging'] = df['service_type'].str.contains('imaging', case=False, na=False).astype(int)

# Outpatient services
df['is_outpatient'] = df['service_type'].str.contains('outpatient|same-day', case=False, na=False).astype(int)

# Surgical procedures
df['is_surgery'] = df['service_type'].str.contains('surgery|procedure', case=False, na=False).astype(int)

# High-complexity DRG (associated with higher billing exposure)
high_complexity_keywords = ['SEPTICEMIA', 'HEART FAILURE', 'INFARCTION', 'HEMORRHAGE',
                             'REPLACEMENT', 'SPINAL', 'VASCULAR', 'PULMONARY EDEMA']
pattern = '|'.join(high_complexity_keywords)
df['high_complexity_drg'] = df['DRG Definition'].str.contains(pattern, case=False, na=False).astype(int)

print('Service flag features created (5 flags):')
for col in ['is_er', 'is_imaging', 'is_outpatient', 'is_surgery', 'high_complexity_drg']:
    print(f'  {col}: {df[col].sum():,} positive ({df[col].mean()*100:.1f}%)')

## 4. Provider-Level Aggregate Features

Provider history captures systematic overbilling patterns at the hospital level.

In [ ]:
print('Computing provider-level aggregates...')

provider_stats = df.groupby('provider_id').agg(
    provider_avg_gap    = ('blended_payment_gap', 'mean'),
    provider_gap_std    = ('blended_payment_gap', 'std'),
    provider_avg_ratio  = ('blended_charge_ratio', 'mean'),
    provider_record_count = ('provider_id', 'count'),
).reset_index()

provider_stats['provider_gap_std'] = provider_stats['provider_gap_std'].fillna(0)

df = df.merge(provider_stats, on='provider_id', how='left')

# Provider risk index: composite score (0-1)
ratio_rank = df['provider_avg_ratio'].rank(pct=True)
gap_rank   = df['provider_avg_gap'].rank(pct=True)
df['provider_risk_index'] = (0.6 * ratio_rank + 0.4 * gap_rank).clip(0, 1)

print(f'Provider aggregates added for {df["provider_id"].nunique()} providers')
print(f'  avg provider_avg_gap:   ${df["provider_avg_gap"].mean():,.0f}')
print(f'  avg provider_avg_ratio: {df["provider_avg_ratio"].mean():.2f}')
print(f'  avg provider_risk_index:{df["provider_risk_index"].mean():.3f}')

## 5. State-Level Aggregate Features

Geographic risk captures regional pricing patterns and state regulatory environments.

In [ ]:
print('Computing state-level aggregates...')

state_stats = df.groupby('provider_state').agg(
    state_avg_gap       = ('blended_payment_gap', 'mean'),
    state_avg_ratio     = ('blended_charge_ratio', 'mean'),
    state_median_payment= ('average_total_payments', 'median'),
    state_record_count  = ('provider_state', 'count'),
).reset_index()

df = df.merge(state_stats, on='provider_state', how='left')

# State risk index: composite score (0-1)
s_ratio_rank = df['state_avg_ratio'].rank(pct=True)
s_gap_rank   = df['state_avg_gap'].rank(pct=True)
df['state_risk_index'] = (0.5 * s_ratio_rank + 0.5 * s_gap_rank).clip(0, 1)

print(f'State aggregates added for {df["provider_state"].nunique()} states')
print(f'  avg state_avg_ratio:  {df["state_avg_ratio"].mean():.2f}')
print(f'  avg state_risk_index: {df["state_risk_index"].mean():.3f}')

## 6. Service Risk Weight

Composite weight based on service type risk factors.

In [ ]:
# Service risk weight: weighted combination of service flags
# ER has highest weight (most likely to have out-of-network providers)
df['service_risk_weight'] = (
    0.35 * df['is_er'] +
    0.25 * df['is_surgery'] +
    0.15 * df['is_imaging'] +
    0.15 * df['high_complexity_drg']
)

print(f'service_risk_weight: mean={df["service_risk_weight"].mean():.3f}')

## 7. Text Feature (DRG Description)

In [ ]:
# Clean DRG text for TF-IDF vectorization in model training
df['drg_text'] = df['DRG Definition'].str.lower().str.replace(r'[^a-z\s]', '', regex=True).str.strip()

print('DRG text feature created')
print('Sample DRG text entries:')
print(df['drg_text'].value_counts().head(5))

## 8. Create Proxy Labels

Since we don't have actual "surprise bill" ground truth, we create proxy labels based on known risk indicators from healthcare research.

In [ ]:
print('Creating proxy labels...')

# === PROXY LABEL 1: Binary classification label ===
# High surprise bill risk if:
# - charge ratio is in top 30% (high overbilling relative to payments)
# - OR payment gap is in top 35%
# - AND provider has above-median risk index
charge_ratio_thresh  = df['blended_charge_ratio'].quantile(0.70)
payment_gap_thresh   = df['blended_payment_gap'].quantile(0.65)
provider_risk_thresh = df['provider_risk_index'].quantile(0.50)

financial_high_risk = (
    (df['blended_charge_ratio'] >= charge_ratio_thresh) |
    (df['blended_payment_gap'] >= payment_gap_thresh)
)

# Service type boost: ER and complex surgery increase risk
service_boost = (df['is_er'] == 1) | (df['high_complexity_drg'] == 1)

df['proxy_surprise_bill_label'] = (
    financial_high_risk &
    (df['provider_risk_index'] >= provider_risk_thresh)
).astype(int)

# Small boost for ER/complex DRG
boost_mask = service_boost & financial_high_risk
df.loc[boost_mask, 'proxy_surprise_bill_label'] = 1

label_rate = df['proxy_surprise_bill_label'].mean()
print(f'  Classification label rate: {label_rate:.1%} positive ({df["proxy_surprise_bill_label"].sum():,} positive cases)')

# === PROXY LABEL 2: Regression target (exposure amount) ===
# Estimated financial exposure = gap between what provider charges vs what insurance covers
df['proxy_exposure_amount'] = np.where(
    df['proxy_surprise_bill_label'] == 1,
    df['blended_payment_gap'] * np.random.uniform(0.15, 0.35, len(df)),
    df['blended_payment_gap'] * np.random.uniform(0.02, 0.10, len(df))
).clip(lower=0)

print(f'  Regression target (exposure): mean=${df["proxy_exposure_amount"].mean():,.0f}, max=${df["proxy_exposure_amount"].max():,.0f}')

## 9. Feature Summary

In [ ]:
feature_cols = [
    'average_covered_charges', 'average_total_payments', 'Billed amount', 'Medicare payment',
    'charge_ratio_inpatient', 'payment_gap_inpatient', 'charge_ratio_outpatient', 'payment_gap_outpatient',
    'blended_charge_ratio', 'blended_payment_gap',
    'log_avg_covered', 'log_avg_payments', 'log_billed_amount', 'log_medicare_payment', 'log_blended_gap',
    'is_er', 'is_imaging', 'is_outpatient', 'is_surgery', 'high_complexity_drg',
    'provider_avg_gap', 'provider_gap_std', 'provider_avg_ratio', 'provider_record_count',
    'state_avg_gap', 'state_avg_ratio', 'state_median_payment', 'state_record_count',
    'provider_risk_index', 'state_risk_index', 'service_risk_weight',
    'provider_state', 'service_type', 'drg_text',
    'proxy_surprise_bill_label', 'proxy_exposure_amount',
]
id_cols = ['provider_id', 'provider_name']

print(f'Total features: {len(feature_cols) - 3 - 2} numeric/categorical + 1 text = {len(feature_cols) - 2} feature columns')
print(f'Target columns: proxy_surprise_bill_label, proxy_exposure_amount')
print(f'\nFeature groups:')
print(f'  Base financial:     4 features')
print(f'  Ratio/gap:          6 features')
print(f'  Log transforms:     5 features')
print(f'  Service flags:      5 features')
print(f'  Provider aggregates:4 features + risk index')
print(f'  State aggregates:   4 features + risk index')
print(f'  Service weight:     1 feature')
print(f'  Categorical:        2 (state, service_type)')
print(f'  Text:               1 (drg_text for TF-IDF)')

In [ ]:
# Visualize label distribution and key features
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Label distribution
label_counts = df['proxy_surprise_bill_label'].value_counts()
axes[0,0].bar(['Low Risk (0)', 'High Risk (1)'], label_counts.values, color=['steelblue', 'tomato'])
axes[0,0].set_title('Proxy Label Distribution')
axes[0,0].set_ylabel('Count')
for i, v in enumerate(label_counts.values):
    axes[0,0].text(i, v + 100, f'{v:,}', ha='center')

# Charge ratio by label
for label in [0, 1]:
    data = df[df['proxy_surprise_bill_label'] == label]['blended_charge_ratio']
    axes[0,1].hist(data.clip(upper=10), bins=40, alpha=0.6, label=f'Label={label}')
axes[0,1].set_title('Charge Ratio by Risk Label')
axes[0,1].set_xlabel('Blended Charge Ratio')
axes[0,1].legend()

# Payment gap distribution
axes[1,0].hist(df['blended_payment_gap'].clip(upper=50000), bins=50, color='mediumorchid', alpha=0.7)
axes[1,0].axvline(df[df['proxy_surprise_bill_label']==1]['blended_payment_gap'].mean(),
                  color='red', linestyle='--', label='High risk mean')
axes[1,0].set_title('Payment Gap Distribution')
axes[1,0].set_xlabel('Blended Payment Gap ($)')
axes[1,0].legend()

# Provider risk index
axes[1,1].hist(df['provider_risk_index'], bins=40, color='darkcyan', alpha=0.7)
axes[1,1].set_title('Provider Risk Index Distribution')
axes[1,1].set_xlabel('Provider Risk Index (0-1)')

plt.tight_layout()
plt.savefig('artifacts/02_feature_distributions.png', dpi=150, bbox_inches='tight')
plt.show()
print('Feature distribution plots saved')

## 10. Save Feature Dataset

In [ ]:
# Save the full feature dataset (include id cols for dashboard analytics)
save_cols = id_cols + feature_cols
model_df = df[save_cols].copy()

output_path = 'data/model_features.parquet'
model_df.to_parquet(output_path, index=False)

print(f'Saved feature dataset to: {output_path}')
print(f'Shape: {model_df.shape}')
print(f'File size: {os.path.getsize(output_path) / 1024:.1f} KB')
print(f'\nLabel distribution:')
print(model_df['proxy_surprise_bill_label'].value_counts())
print(f'\nExposure amount summary:')
print(model_df['proxy_exposure_amount'].describe())
print(f'\n✅ Notebook 2 complete. Run Notebook 3 (03_model_training_RF_ONLY.ipynb) next.')